# Build WRF-Hydro / CESM

Setup, build, and run the coupled WRF-Hydro + CTSM case described in the top-level
`README.md`, driven from a notebook so it can be run on
[NCAR JupyterHub](https://jupyterhub.hpc.ucar.edu) with `$SCRATCH` visible.

Two backends are supported:

| Backend | Where | What works |
|---|---|---|
| `derecho` | NCAR JupyterHub (Derecho/Casper session) | everything: `create_newcase --mach derecho`, `case.build`, `case.submit` |
| `docker` | local Mac/Linux with the `cesm-wrf-hydro` image | source checkout, module/library sanity checks, experimenting with the build |

**Docker does not run on NCAR JupyterHub.** JupyterHub sessions are ordinary
unprivileged user processes inside a PBS job; there is no Docker daemon and no root,
so `docker run` will fail there. The container runtime available on Derecho/Casper is
Apptainer, and this image would have to be converted to a `.sif` first — see the last
section. You do not need it: Derecho already provides the full compiler/ESMF stack
through `modules/intel-cesm`, and CIME has its own stack for `--mach` anyway
against. The notebook's Jupyter kernel does *not* need to be able to build anything —
every build step is shelled out to a login shell that loads the Derecho modules
itself.

## 1. Configuration

Edit this cell. Everything downstream is derived from it.

In [1]:
import os
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

# --- machine -------------------------------------------------------------
# LMOD_SYSTEM_NAME is set by the system Lmod config on both Derecho and Casper.
# Override it here if you want to target the other one.
MACHINE = os.environ.get("LMOD_SYSTEM_NAME") or os.environ.get("NCAR_HOST") or "derecho"

# --- case settings (mirror of the top-level Makefile) ---------------------
CASE_NAME = f"hydro-test-{MACHINE}"
COMPILER  = "intel"
COMPSET   = "I2000Ctsm50NwpSpNldasWRFHydro"
RES       = "nldas2_rnldas2_mnldas2"
PROJECT   = "NWCA0002"

# --- environment settings ------------------------------------------------
# modules/intel-cesm/25.12.lua; picks cray-mpich or openmpi by machine.
# COMPILER above is what CIME builds with, and CIME loads its own stack --
# these two are independent, see section 4.
MODULE_NAME  = "intel-cesm"
DOCKER_IMAGE = "cesm-wrf-hydro"

# --- locate the repository root ------------------------------------------
REPO = Path.cwd().resolve()
while not (REPO / "src" / "ctsm").is_dir():
    if REPO == REPO.parent:
        raise RuntimeError(
            "Could not find the wrf-hydro_cesm root above %s. "
            "Start the notebook from inside the repository." % Path.cwd()
        )
    REPO = REPO.parent

# --- scratch / case directories ------------------------------------------
# On Derecho and Casper $SCRATCH is set by ncarenv. Fall back to the
# canonical path so the cell still resolves in a bare login shell.
SCRATCH = Path(os.environ.get("SCRATCH") or f"/glade/derecho/scratch/{os.environ.get('USER', '')}")
CASE_DIR = SCRATCH / "cases" / CASE_NAME

PESFILE = REPO / "src/ctsm/components/wrfhydro/src/CPL/CESM_cpl/cime_config/config_pes.xml"
SCRIPTS = REPO / "src/ctsm/cime/scripts"

print(f"machine  {MACHINE} ({COMPILER}, modules/{MODULE_NAME})")
print(f"repo     {REPO}")
print(f"scratch  {SCRATCH}")
print(f"case dir {CASE_DIR}")
print(f"pesfile  {PESFILE}  (exists: {PESFILE.is_file()})")

machine  casper (intel, modules/intel-cesm)
repo     /glade/work/soren/src/ctsm/wrf-hydro_ctsm
scratch  /glade/derecho/scratch/soren
case dir /glade/derecho/scratch/soren/cases/hydro-test-casper
pesfile  /glade/work/soren/src/ctsm/wrf-hydro_ctsm/src/ctsm/components/wrfhydro/src/CPL/CESM_cpl/cime_config/config_pes.xml  (exists: True)


## 2. Pick a backend

`derecho` whenever `/glade` is mounted (that is, inside a JupyterHub session or on a
login node); otherwise fall back to the Docker image for local development.

In [2]:
ON_GLADE = Path("/glade").is_dir()
HAVE_DOCKER = shutil.which("docker") is not None

if ON_GLADE:
    BACKEND = "derecho"
elif HAVE_DOCKER:
    BACKEND = "docker"
else:
    BACKEND = None

print(f"/glade mounted : {ON_GLADE}")
print(f"docker present : {HAVE_DOCKER}")
print(f"backend        : {BACKEND}")

if BACKEND == "docker":
    print(
        "\nLocal mode. `--mach derecho` needs Derecho's module stack, PBS, and the\n"
        "/glade input data, so the case steps (sections 5-9) are Derecho-only.\n"
        "Sections 3-4 (environment check) and the source update work here."
    )
elif BACKEND is None:
    raise RuntimeError("Neither /glade nor docker is available; nothing to run against.")

/glade mounted : True
docker present : False
backend        : derecho


## 3. Command runner

Build steps run in a **fresh `bash -l` subprocess**, not in the kernel's own
environment. That matters on JupyterHub: the kernel is started inside a conda
environment (NPL or similar), and its `PYTHONPATH` / `CONDA_PREFIX` would otherwise
leak into CIME. Those variables are stripped before the shell starts.

Module loading follows the README order — `module use modules`, `module purge`,
`module load {MODULE_NAME}`.

In [3]:
# Variables inherited from the Jupyter kernel that must not leak into the build.
#
# CLICOLOR_FORCE is not cosmetic: CIME builds Macros.make by parsing the output of
# `cmake -DCONVERT_TO_MAKE=ON`, and with colour forced, cmake wraps every message()
# line in \033[0m. make then sees a variable called "\033[0mNETCDF_PATH", so
# Tools/Makefile reports "NETCDF not found" and csm_share fails to build.
_STRIP_PREFIXES = (
    "PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP",
    "CONDA_", "VIRTUAL_ENV", "_CE_",
    "JPY_", "JUPYTER_",
    "CLICOLOR", "FORCE_COLOR", "COLORTERM",
)

MODULE_PRELUDE = f"""
module use {shlex.quote(str(REPO / 'modules'))}
module purge
module load {MODULE_NAME}
""".strip()


def _clean_env():
    return {k: v for k, v in os.environ.items() if not k.startswith(_STRIP_PREFIXES)}


def _docker_path(path):
    """Map a host path into the container's /src mount."""
    path = Path(path).resolve()
    try:
        return "/src" if path == REPO else "/src/" + str(path.relative_to(REPO))
    except ValueError:
        raise RuntimeError(f"{path} is outside {REPO}; not visible in the container.")


def run(cmd, cwd=None, modules=True, check=True):
    """Run `cmd` in a login shell and stream its output into the notebook.

    On the derecho backend the shell is local and loads modules/{MODULE_NAME}.
    On the docker backend the shell runs inside DOCKER_IMAGE with the repo
    bind-mounted at /src.
    """
    cwd = Path(cwd) if cwd else REPO
    script = f"set -o pipefail\n{MODULE_PRELUDE if modules and BACKEND == 'derecho' else ''}\n{cmd}"

    if BACKEND == "docker":
        # The ESMF base image sets ENTRYPOINT ["/bin/bash", "-l"], which would
        # swallow the command; override it so `-lc <script>` is what bash sees.
        argv = [
            "docker", "run", "--rm",
            "-v", f"{REPO}:/src",
            "-w", _docker_path(cwd),
            "--entrypoint", "/bin/bash",
            DOCKER_IMAGE, "-lc", script,
        ]
        env = None
    else:
        argv = ["bash", "-lc", script]
        env = _clean_env()

    print(f"$ ({BACKEND}:{cwd}) {cmd}\n", flush=True)
    proc = subprocess.Popen(
        argv, cwd=None if BACKEND == "docker" else str(cwd), env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    if check and rc != 0:
        raise RuntimeError(f"command failed with exit code {rc}: {cmd}")
    return rc

## 4. Environment check

Two different environments are in play here, and it is worth being explicit about which
is which:

| | Set by | Used by |
|---|---|---|
| **Notebook shell** | `MODULE_PRELUDE` -> `modules/{MODULE_NAME}` | anything you run by hand: `nc-config`, a one-off compile, the first check below |
| **Case build** | CIME's `env_mach_specific` | `case.setup`, `case.build`, `case.submit` |

They are independent. CIME does `load ncarenv/...`, then `purge`, then loads the stack
from `ccs_config/machines/<MACHINE>` — so it discards whatever the notebook loaded. A
green light in the first check says nothing about the second.

The second check reads `.env_mach_specific.sh`, which `case.setup` writes, so it only
works once section 7 has run. That is the one to trust before committing to a long
build.

One thing to remember throughout: **each `run()` is a fresh login shell.**
`run("module load X")` followed by `run("module list")` will show nothing, because the
first shell exited. Anything that depends on a module has to happen in the same call.

In [4]:
# What the notebook's own commands get: MODULE_PRELUDE -> modules/{MODULE_NAME}.
# Everything in one run() call, because each call is a separate shell.
_TOOLS = "mpif90 ifx ifort gfortran nc-config nf-config python3"

if BACKEND == "derecho":
    run(
        "module list\n"
        "echo\n"
        f"for t in {_TOOLS}; do\n"
        "  printf '  %-10s %s\\n' \"$t\" \"$(command -v $t || echo -- MISSING)\"\n"
        "done\n"
        "echo\n"
        "echo \"  ESMFMKFILE=${ESMFMKFILE:-<unset>}\"\n"
        "command -v mpif90 >/dev/null && mpif90 --version | head -1\n"
        "command -v nc-config >/dev/null && echo \"  netcdf $(nc-config --version)\"\n"
        "true"
    )
else:
    run(
        f"for t in {_TOOLS} pnetcdf_version; do\n"
        "  printf '  %-10s %s\\n' \"$t\" \"$(command -v $t || echo -- MISSING)\"\n"
        "done\n"
        "echo \"  ESMFMKFILE=${ESMFMKFILE:-<unset>}\"\n"
        "nc-config --version\n"
        "true",
        modules=False,
    )

$ (derecho:/glade/work/soren/src/ctsm/wrf-hydro_ctsm) module list
echo
for t in mpif90 ifx ifort gfortran nc-config nf-config python3; do
  printf '  %-10s %s\n' "$t" "$(command -v $t || echo -- MISSING)"
done
echo
echo "  ESMFMKFILE=${ESMFMKFILE:-<unset>}"
command -v mpif90 >/dev/null && mpif90 --version | head -1
command -v nc-config >/dev/null && echo "  netcdf $(nc-config --version)"
true

The following modules were not unloaded:
  (Use "module --force purge" to unload all):

  1) ncarenv/25.10

Currently Loaded Modules:
  1) ncarenv/25.10  (S)   5) ucx/1.19.0         9) parallel-netcdf/1.14.1
  2) intel/2025.3.2       6) openmpi/5.0.9     10) parallelio/2.6.8
  3) cuda/12.9.0          7) hdf5-mpi/1.14.6   11) esmf-mpi/8.9.1
  4) gcc/15.1.0           8) netcdf-mpi/4.9.3  12) intel-cesm/25.12

  Where:
   S:  Module is Sticky, requires --force to unload or purge

 


  mpif90     /glade/u/apps/casper/25.10/spack/opt/spack/openmpi/5.0.9/intel-oneapi-compilers/2025.3.2/2l3x/bin/mpif90

In [5]:
# What case.build actually uses. modules=False on purpose: we want CIME's environment,
# not MODULE_PRELUDE's. Needs section 7 to have run.
_env_sh = CASE_DIR / ".env_mach_specific.sh"

if not _env_sh.is_file():
    print(f"{_env_sh}\n  not written yet -- run sections 6 and 7, then come back to this cell.")
else:
    run(
        "source .env_mach_specific.sh\n"
        "module list\n"
        "echo\n"
        "echo \"  ESMFMKFILE=${ESMFMKFILE:-<unset>}\"\n"
        "echo \"  mpif90     $(command -v mpif90)\"\n"
        "mpif90 --version | head -1\n"
        "true",
        cwd=CASE_DIR,
        modules=False,
    )

$ (derecho:/glade/derecho/scratch/soren/cases/hydro-test-casper) source .env_mach_specific.sh
module list
echo
echo "  ESMFMKFILE=${ESMFMKFILE:-<unset>}"
echo "  mpif90     $(command -v mpif90)"
mpif90 --version | head -1
true


Due to MODULEPATH changes, the following have been reloaded:
  1) conda/latest

The following have been reloaded with a version change:
  1) ncarenv/25.10 => ncarenv/24.12

The following modules were not unloaded:
  (Use "module --force purge" to unload all):

  1) ncarenv/24.12

Currently Loaded Modules:
  1) ncarenv/24.12         (S)   8) hdf5-mpi/1.12.3
  2) cmake/3.31.0                9) netcdf-mpi/4.9.2
  3) intel-oneapi/2024.2.1      10) parallel-netcdf/1.14.0
  4) mkl/2024.2.2               11) parallelio/2.6.5
  5) cuda/12.3.2                12) esmf/8.8.0
  6) ucx/1.17.0                 13) ncarcompilers/1.0.0
  7) openmpi/5.0.6

  Where:
   S:  Module is Sticky, requires --force to unload or purge

 


  ESMFMKFILE=/glade/u/apps/casper/24.12/spack/opt

## 5. Update the source tree

Off by default — it mutates the checkout. Note that CTSM's submodules are managed by
`git-fleximod`, so `git submodule update --init` must **not** be recursive.

In [6]:
UPDATE_SOURCE = False

if UPDATE_SOURCE:
    run("git submodule update --init", cwd=REPO)
    run("./bin/git-fleximod status", cwd=REPO / "src/ctsm")
    run("./bin/git-fleximod update", cwd=REPO / "src/ctsm")
else:
    print("skipped; set UPDATE_SOURCE = True to sync submodules")

skipped; set UPDATE_SOURCE = True to sync submodules


## 6. Create the case

Equivalent to `make setup`. `create_newcase` refuses to overwrite an existing case
directory, so remove it first if you are starting over.

In [7]:
assert BACKEND == "derecho", "case creation requires the Derecho machine config and /glade"

if CASE_DIR.exists():
    print(f"{CASE_DIR} already exists -- skipping create_newcase.")
    print("To start over:  shutil.rmtree(CASE_DIR)  (and remove the bld/run dirs under $SCRATCH)")
else:
    run(
        "./create_newcase"
        f" --case {shlex.quote(str(CASE_DIR))}"
        f" --mach {MACHINE}"
        f" --compiler {COMPILER}"
        f" --compset {COMPSET}"
        f" --res {RES}"
        " --run-unsupported"
        f" --project {PROJECT}"
        f" --pesfile {shlex.quote(str(PESFILE))}",
        cwd=SCRIPTS,
    )

/glade/derecho/scratch/soren/cases/hydro-test-casper already exists -- skipping create_newcase.
To start over:  shutil.rmtree(CASE_DIR)  (and remove the bld/run dirs under $SCRATCH)


## 7. Configure and set up the case

One-hour run, hourly river-routing coupling, then `case.setup`.

In [8]:
run("./xmlchange STOP_OPTION=nhours,STOP_N=1,ROF_NCPL=24", cwd=CASE_DIR)
run("./case.setup", cwd=CASE_DIR)

$ (derecho:/glade/derecho/scratch/soren/cases/hydro-test-casper) ./xmlchange STOP_OPTION=nhours,STOP_N=1,ROF_NCPL=24

The following modules were not unloaded:
  (Use "module --force purge" to unload all):

  1) ncarenv/25.10
$ (derecho:/glade/derecho/scratch/soren/cases/hydro-test-casper) ./case.setup

The following modules were not unloaded:
  (Use "module --force purge" to unload all):

  1) ncarenv/25.10
Setting resource.RLIMIT_STACK to -1 from (-1, -1)
Machine/Decomp/Pes configuration has already been done ...skipping
If an old case build already exists, might want to run 'case.build --clean' before building
You can now run './preview_run' to get more info on how your case will be run


0

In [9]:
# Optional: inspect the namelists CIME generated.
run("./preview_namelists", cwd=CASE_DIR)

$ (derecho:/glade/derecho/scratch/soren/cases/hydro-test-casper) ./preview_namelists

The following modules were not unloaded:
  (Use "module --force purge" to unload all):

  1) ncarenv/25.10
Setting resource.RLIMIT_STACK to -1 from (-1, -1)
  2026-09-13 10:44:03 atm 
Create namelist for component datm
   Calling /glade/work/soren/src/ctsm/wrf-hydro_ctsm/src/ctsm/components/cdeps/datm/cime_config/buildnml
  2026-09-13 10:44:03 lnd 
Create namelist for component clm
   Calling /glade/work/soren/src/ctsm/wrf-hydro_ctsm/src/ctsm/cime_config/buildnml
IMPORTANT NOTE: LND_TUNING_MODE is clm5_0_NLDAS2 which does NOT have tuned settings, so using the closest option which is clm5_0_GSWP3v1
              : To suppress this message explicitly set LND_TUNING_MODE=clm5_0_NLDAS2 for your case
  2026-09-13 10:44:03 ice 
Create namelist for component sice
   Calling /glade/work/soren/src/ctsm/wrf-hydro_ctsm/src/ctsm/cime/CIME/non_py/src/components/stub_comps_nuopc/sice/cime_config/buildnml
  2026-09-

0

## 8. Build

A CESM build is long and heavily parallel. Under JupyterHub you are already on a
compute node, so building in place is fine. From a Derecho **login** node, set
`USE_QCMD = True` to push the build onto a batch node instead.

In [10]:
USE_QCMD = False

build_cmd = "./case.build --verbose"
if USE_QCMD:
    build_cmd = f"qcmd -A {PROJECT} -- {build_cmd}"

run(build_cmd, cwd=CASE_DIR)

$ (derecho:/glade/derecho/scratch/soren/cases/hydro-test-casper) ./case.build --verbose

The following modules were not unloaded:
  (Use "module --force purge" to unload all):

  1) ncarenv/25.10
09-13 10:44 CIME.build   INFO     Building case in directory /glade/derecho/scratch/soren/cases/hydro-test-casper
09-13 10:44 CIME.build   INFO     sharedlib_only is False
09-13 10:44 CIME.build   INFO     model_only is False
09-13 10:44 CIME.XML.env_mach_specific INFO     Setting resource.RLIMIT_STACK to -1 from (-1, -1)
09-13 10:44 CIME.build   INFO     Generating component namelists as part of build
09-13 10:44 CIME.case.preview_namelists INFO       2026-09-13 10:44:06 atm 
09-13 10:44 CIME.case.preview_namelists INFO     Create namelist for component datm
09-13 10:44 CIME.utils   INFO        Calling /glade/work/soren/src/ctsm/wrf-hydro_ctsm/src/ctsm/components/cdeps/datm/cime_config/buildnml
09-13 10:44 CIME.case.preview_namelists INFO       2026-09-13 10:44:06 lnd 
09-13 10:44 CIME.case.p

0

## 9. First run

Per the README, the very first run must have WRF-Hydro on a single process so that
`run/DOMAIN/geo_em.d01.nc` gets created; afterwards `NTASKS_ROF` can go back to 8.
The rest of the job still uses all eight PETs.

In [11]:
FIRST_RUN = True   # set False once run/DOMAIN/geo_em.d01.nc exists

geo_em = CASE_DIR / "run" / "DOMAIN" / "geo_em.d01.nc"
ntasks_rof = 1 if FIRST_RUN and not geo_em.is_file() else 8

print(f"geo_em.d01.nc present: {geo_em.is_file()}  ->  NTASKS_ROF={ntasks_rof}")
run(f"./xmlchange NTASKS_ROF={ntasks_rof}", cwd=CASE_DIR)
run("./case.setup --reset", cwd=CASE_DIR)
run("./xmlquery NTASKS_ROF", cwd=CASE_DIR)

# case.setup --reset always sets BUILD_COMPLETE=False (case_setup.py:288, "rebuild the
# models (even on restart)"), whether or not the PE layout actually changed. Without
# this, case.submit refuses with "Build complete is not True". Nothing needs
# recompiling, so it is a quick relink.
run("./case.build", cwd=CASE_DIR)
run("./xmlquery BUILD_COMPLETE", cwd=CASE_DIR)

geo_em.d01.nc present: False  ->  NTASKS_ROF=1
$ (derecho:/glade/derecho/scratch/soren/cases/hydro-test-casper) ./xmlchange NTASKS_ROF=1

The following modules were not unloaded:
  (Use "module --force purge" to unload all):

  1) ncarenv/25.10
$ (derecho:/glade/derecho/scratch/soren/cases/hydro-test-casper) ./case.setup --reset

The following modules were not unloaded:
  (Use "module --force purge" to unload all):

  1) ncarenv/25.10
Successfully cleaned .case.run
Successfully cleaned env_mach_specific.xml
Successfully cleaned Macros.make
Successfully cleaned Macros.cmake
Successfully cleaned cmake_macros
Setting resource.RLIMIT_STACK to -1 from (-1, -1)


job is case.run USER_REQUESTED_WALLTIME None USER_REQUESTED_QUEUE None WALLTIME_FORMAT %H:%M:%S
Creating batch scripts
Writing case.run script from input template /glade/work/soren/src/ctsm/wrf-hydro_ctsm/src/ctsm/ccs_config/machines/template.case.run
Creating file .case.run
Writing case.st_archive script from input template /glade/

0

In [12]:
# case.submit has a built-in non-batch mode -- nothing in the script needs changing.
# --no-batch: "Do not submit jobs to batch system, run locally." It runs
#   mpirun -np $TOTALPES .../cesm.exe  in this process instead of qsub'ing .case.run,
# so the cell blocks until the run finishes.
#
# Note the run is quiet: preview_run shows the model's stdout is redirected to
# run/cesm.log.$LID, not to the terminal. Use the next cell to look at it.
INTERACTIVE = True     # False to queue it with qsub instead

run("./case.submit --no-batch" if INTERACTIVE else "./case.submit", cwd=CASE_DIR)

$ (derecho:/glade/derecho/scratch/soren/cases/hydro-test-casper) ./case.submit --no-batch

The following modules were not unloaded:
  (Use "module --force purge" to unload all):

  1) ncarenv/25.10
Setting resource.RLIMIT_STACK to -1 from (-1, -1)
  2026-09-13 10:45:16 atm 
Create namelist for component datm
   Calling /glade/work/soren/src/ctsm/wrf-hydro_ctsm/src/ctsm/components/cdeps/datm/cime_config/buildnml
  2026-09-13 10:45:16 lnd 
Create namelist for component clm
   Calling /glade/work/soren/src/ctsm/wrf-hydro_ctsm/src/ctsm/cime_config/buildnml
IMPORTANT NOTE: LND_TUNING_MODE is clm5_0_NLDAS2 which does NOT have tuned settings, so using the closest option which is clm5_0_GSWP3v1
              : To suppress this message explicitly set LND_TUNING_MODE=clm5_0_NLDAS2 for your case
  2026-09-13 10:45:16 ice 
Create namelist for component sice
   Calling /glade/work/soren/src/ctsm/wrf-hydro_ctsm/src/ctsm/cime/CIME/non_py/src/components/stub_comps_nuopc/sice/cime_config/buildnml
  202

0

In [13]:
# Job status and the tail of the run log.
run("qstat -u $USER || true", cwd=CASE_DIR, check=False)
run("ls -lt run/ | head -20", cwd=CASE_DIR, check=False)
run("tail -40 $(ls -t run/cesm.log.* 2>/dev/null | head -1) || echo 'no cesm.log yet'",
    cwd=CASE_DIR, check=False)

$ (derecho:/glade/derecho/scratch/soren/cases/hydro-test-casper) qstat -u $USER || true

The following modules were not unloaded:
  (Use "module --force purge" to unload all):

  1) ncarenv/25.10

casper-pbs:
                                                            Req'd  Req'd   Elap 
Job ID          Username Queue    Jobname    SessID NDS TSK Memory Time  S Time 
--------------- -------- -------- ---------- ------ --- --- ------ ----- - -----
5845813.casper* soren    jhublog* sys-dashb*  37341   1   2   10gb 720:0 R 142:5
$ (derecho:/glade/derecho/scratch/soren/cases/hydro-test-casper) ls -lt run/ | head -20

The following modules were not unloaded:
  (Use "module --force purge" to unload all):

  1) ncarenv/25.10
ls: cannot access 'run/': No such file or directory
$ (derecho:/glade/derecho/scratch/soren/cases/hydro-test-casper) tail -40 $(ls -t run/cesm.log.* 2>/dev/null | head -1) || echo 'no cesm.log yet'

The following modules were not unloaded:
  (Use "module --force purge" t

0

## Containers on NCAR JupyterHub

Short answer: **no Docker.** A JupyterHub session is a PBS job running as you on a
Casper or Derecho node. Docker needs a root-owned daemon, which HPC systems do not
give users, so `docker run` will not work from a notebook there. The `docker` backend
above is for your Mac only.

If you do want a container on Derecho, the supported runtime is Apptainer:

```bash
module avail apptainer          # confirm it is there
```

Getting this image over is the awkward part — Apptainer reads a `.sif`, not your Mac's
local Docker daemon. Either push to a registry and pull on Derecho:

```bash
# on the Mac -- note --platform: Derecho is x86_64, Apple silicon is arm64
docker buildx build --platform linux/amd64 -t <registry>/cesm-wrf-hydro:latest src/docker
docker push <registry>/cesm-wrf-hydro:latest

# on Derecho
apptainer pull cesm-wrf-hydro.sif docker://<registry>/cesm-wrf-hydro:latest
apptainer exec --bind $SCRATCH cesm-wrf-hydro.sif bash -lc '...'
```

or build the `.sif` locally (`apptainer build ... docker-daemon://cesm-wrf-hydro:latest`,
needs Apptainer on the Mac, i.e. inside a Linux VM) and `scp` it over.

**But you almost certainly should not.** The container is not a shortcut around a
missing Jupyter environment, because the Jupyter environment is not what builds this
project — `case.build` does, in a subprocess, using whatever the Derecho modules
provide. Section 3's runner is exactly that. And on Derecho the container is strictly
worse: it has no `derecho` machine config, no PBS, no Cray MPI, and no `/glade`
inputdata. The `container` machine that ships in
`src/ctsm/ccs_config/machines/container/` also does not match this image — it expects
mpich under `/usr/local`, while `src/docker/Dockerfile` builds on ESMF's OpenMPI spack
view — so a full in-container case build would need a new machine config that does not
exist in this repo yet.